In [1]:
import pandas as pd
import xml.etree.ElementTree as ET

def rss_to_dataframe(source):
    if source.startswith("http://") or source.startswith("https://"):
        import urllib.request
        with urllib.request.urlopen(source) as response:
            tree = ET.parse(response)
    else:
        tree = ET.parse(source)

    root = tree.getroot()
    
    channel = root.find("channel")
    items = channel.findall("item")

    records = []
    for item in items:
        record = {}
        for child in item:
            tag = child.tag.split("}")[-1] if "}" in child.tag else child.tag
            record[tag] = child.text
        records.append(record)

    return pd.DataFrame(records)

In [2]:
#Paste Google News RSS file URL & Topic
url = ""
topic = ""

df = rss_to_dataframe(url)
df['pubDate'] = pd.to_datetime(df['pubDate']).dt.strftime('%Y-%m-%d %H:%M:%S')
df = df[['pubDate', 'title']]

In [3]:
df.to_csv(f'{topic}_data.csv', index=False)